# Lab 03: Image Gradients, Finite Differences, and 2D Convolution

[Back to Session Overview](../README.md) · [Module 02 Index](../../README.md)

---

## Overview

In computer vision and image processing, a digital image is represented as a discrete 2D scalar field $I(x, y)$ of pixel intensities. Visual features such as boundaries, contours, textures, and edges correspond to sharp spatial changes in image intensity.

Because digital images are discrete, spatial differentiation cannot take an infinitesimal limit $\Delta x \to 0$. Instead, we evaluate **discrete difference operators**:
1. **Discrete Image Representations:** Inspecting grayscale $(H, W)$ and RGB $(H, W, 3)$ structures in pure Python and NumPy.
2. **Horizontal and Vertical Spatial Gradients:** Approximating directional derivatives using forward and central finite difference schemes without convolution.
3. **Gradient Magnitude & Angle:** Computing $G(x, y) = \sqrt{G_x^2 + G_y^2}$ and $\theta(x, y) = \arctan2(G_y, G_x)$.
4. **2D Discrete Convolution:** Implementing spatial filtering using 2D difference kernels ($x$-kernel and $y$-kernel).
5. **The Sobel Operator:** Exploring separable $3 \times 3$ smoothing + differentiation kernels, verifying manual lecture hand-calculations ($a = -211 \implies |a| = 211$), and benchmarking against OpenCV.


In [ ]:
import math
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = False

---

## 1. Digital Image Representation

A grayscale image is an $H \times W$ matrix where each pixel is an 8-bit unsigned integer $p \in [0, 255]$:
- $0$: Absolute black
- $255$: Pure white

A color image is an $H \times W \times 3$ tensor with red, green, and blue intensity channels.

Here we load `../data/cat.jpg` and convert it into a pure Python 2D list `list[list[int]]` to demonstrate first-principles algorithms before using vectorized libraries.


In [ ]:
data_path = Path("../data/cat.jpg")
assert data_path.is_file(), f"Image fixture not found at {data_path.resolve()}"

# Load image in grayscale and color modes using OpenCV
img_bgr = cv2.imread(str(data_path), cv2.IMREAD_COLOR)
img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
img_gray = cv2.imread(str(data_path), cv2.IMREAD_GRAYSCALE)

print(f"Color Image Shape:     {img_rgb.shape} (dtype: {img_rgb.dtype})")
print(f"Grayscale Image Shape: {img_gray.shape} (dtype: {img_gray.dtype})")
print(f"Intensity Range:       [{img_gray.min()}, {img_gray.max()}]")

# Convert to pure-Python nested 2D list for first-principles algorithmic implementation
cat_image_list = img_gray.tolist()
assert isinstance(cat_image_list, list)
assert isinstance(cat_image_list[0], list)
print(f"Pure Python List Size: {len(cat_image_list)} rows x {len(cat_image_list[0])} cols")

---

## 2. Spatial Derivatives Without Convolution (Nested Loops)

On a continuous function, $f'(x) = \lim_{\Delta x \to 0} \frac{f(x+\Delta x) - f(x-\Delta x)}{2\Delta x}$. On a discrete image grid with unit pixel spacing $\Delta x = 1$, the central difference across symmetric neighbors is:

$$G_x(i, j) = I(i, j+1) - I(i, j-1)$$
$$G_y(i, j) = I(i+1, j) - I(i-1, j)$$

The combined **gradient magnitude** measures total edge strength:

$$G(i, j) = \sqrt{G_x(i, j)^2 + G_y(i, j)^2}$$


In [ ]:
def compute_x_derivative(image):
    """Compute horizontal image derivative G_x via central difference."""
    h = len(image)
    w = len(image[0])
    x_derivative = [[0.0] * w for _ in range(h)]
    for i in range(1, h - 1):
        for j in range(1, w - 1):
            x_derivative[i][j] = float(image[i][j + 1] - image[i][j - 1])
    return x_derivative


def compute_y_derivative(image):
    """Compute vertical image derivative G_y via central difference."""
    h = len(image)
    w = len(image[0])
    y_derivative = [[0.0] * w for _ in range(h)]
    for i in range(1, h - 1):
        for j in range(1, w - 1):
            y_derivative[i][j] = float(image[i + 1][j] - image[i - 1][j])
    return y_derivative


def compute_gradient_magnitude(image):
    """Compute overall edge magnitude G = sqrt(G_x^2 + G_y^2)."""
    x_d = compute_x_derivative(image)
    y_d = compute_y_derivative(image)
    h = len(image)
    w = len(image[0])
    mag = [[0.0] * w for _ in range(h)]
    for i in range(1, h - 1):
        for j in range(1, w - 1):
            mag[i][j] = math.sqrt(x_d[i][j] ** 2 + y_d[i][j] ** 2)
    return mag, x_d, y_d


grad_mag_list, gx_list, gy_list = compute_gradient_magnitude(cat_image_list)
gx_arr = np.array(gx_list)
gy_arr = np.array(gy_list)
mag_arr = np.array(grad_mag_list)

print(f"G_x Range:       [{gx_arr.min():.1f}, {gx_arr.max():.1f}]")
print(f"G_y Range:       [{gy_arr.min():.1f}, {gy_arr.max():.1f}]")
print(f"Magnitude Range: [{mag_arr.min():.1f}, {mag_arr.max():.1f}]")

---

## 3. Spatial Differentiation Using 2D Convolution

Computing discrete differences by accessing specific array neighbors is mathematically equivalent to **2D cross-correlation / convolution** with small difference kernels:

- Horizontal kernel $K_x$:
  $$K_x = \begin{bmatrix} 0 & 0 & 0 \\ -1 & 0 & 1 \\ 0 & 0 & 0 \end{bmatrix}$$

- Vertical kernel $K_y$:
  $$K_y = \begin{bmatrix} 0 & -1 & 0 \\ 0 & 0 & 0 \\ 0 & 1 & 0 \end{bmatrix}$$

When sliding $K_x$ over window $I[i-1:i+2, j-1:j+2]$, the sum of products yields:
$$(-1) \cdot I[i, j-1] + (0) \cdot I[i, j] + (1) \cdot I[i, j+1] = I[i, j+1] - I[i, j-1]$$


In [ ]:
def compute_gradient_using_conv(image, kernel):
    """Convolve a 2D image list with an arbitrary 2D kernel."""
    h = len(image)
    w = len(image[0])
    kh = len(kernel)
    kw = len(kernel[0])
    pad_h = kh // 2
    pad_w = kw // 2
    output = [[0.0] * w for _ in range(h)]
    for i in range(pad_h, h - pad_h):
        for j in range(pad_w, w - pad_w):
            val = 0.0
            for ki in range(kh):
                for kj in range(kw):
                    val += kernel[ki][kj] * image[i - pad_h + ki][j - pad_w + kj]
            output[i][j] = val
    return output


x_kernel = [[0, 0, 0], [-1, 0, 1], [0, 0, 0]]
y_kernel = [[0, -1, 0], [0, 0, 0], [0, 1, 0]]

conv_gx = compute_gradient_using_conv(cat_image_list, x_kernel)
conv_gy = compute_gradient_using_conv(cat_image_list, y_kernel)

# Verify exact mathematical equivalence between direct loop and convolution
conv_gx_arr = np.array(conv_gx)
conv_gy_arr = np.array(conv_gy)

assert np.allclose(gx_arr, conv_gx_arr), "Direct G_x and Convolution G_x must match exactly!"
assert np.allclose(gy_arr, conv_gy_arr), "Direct G_y and Convolution G_y must match exactly!"
print("Mathematical parity confirmed: Direct discrete difference == 2D convolution!")

---

## 4. The Sobel Operator & Hand-Calculation Verification

Simple difference kernels are highly sensitive to high-frequency image noise. The **Sobel operator** suppresses noise by combining discrete differentiation along one axis with triangular smoothing along the orthogonal axis:

$$K_x = \begin{bmatrix} 1 \\ 2 \\ 1 \end{bmatrix} \begin{bmatrix} -1 & 0 & 1 \end{bmatrix} = \begin{bmatrix} -1 & 0 & 1 \\ -2 & 0 & 2 \\ -1 & 0 & 1 \end{bmatrix}$$

$$K_y = \begin{bmatrix} -1 \\ 0 \\ 1 \end{bmatrix} \begin{bmatrix} 1 & 2 & 1 \end{bmatrix} = \begin{bmatrix} -1 & -2 & -1 \\ 0 & 0 & 0 \\ 1 & 2 & 1 \end{bmatrix}$$

### Verification of Lecture Slide Hand Calculation (Slide 69)

In the lecture slides, an image patch is convolved with the Sobel $x$-kernel:
$$\text{Input Patch} = \begin{bmatrix} 230 & \cdot & 147 \\ 237 & \cdot & 188 \\ 210 & \cdot & 180 \end{bmatrix}$$

The linear combination is:
$$a = -230 + 147 - 2(237) + 2(188) - 210 + 180 = -230 + 147 - 474 + 376 - 210 + 180 = -211$$
$$f(a) = |a| = 211$$


In [ ]:
# Hand calculation test fixture
sobel_x = [[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]
sobel_y = [[-1, -2, -1], [0, 0, 0], [1, 2, 1]]

patch = [[230, 0, 147], [237, 0, 188], [210, 0, 180]]

# Compute Sobel x-response on patch center
a = 0
for r in range(3):
    for c in range(3):
        a += sobel_x[r][c] * patch[r][c]

print(f"Calculated Sobel response a: {a}")
print(f"Absolute response |a|:       {abs(a)}")

assert a == -211, f"Expected a = -211, got {a}"
assert abs(a) == 211, f"Expected |a| = 211, got {abs(a)}"
print("Exact match with lecture slide calculation confirmed!")

---

## 5. Visualizing Image Gradients & Benchmarking with OpenCV

Here we compute full-image Sobel filtering using OpenCV's optimized C++ routines and visualize the directional gradients alongside the edge magnitude.


In [ ]:
# OpenCV Sobel gradients (64-bit float precision to preserve negative gradients)
sobel_x_cv = cv2.Sobel(img_gray, cv2.CV_64F, 1, 0, ksize=3)
sobel_y_cv = cv2.Sobel(img_gray, cv2.CV_64F, 0, 1, ksize=3)
sobel_mag_cv = np.sqrt(sobel_x_cv**2 + sobel_y_cv**2)

# Normalize for 8-bit display:
# For directional gradients, shift level by +128 so 0 appears middle gray
gx_display = cv2.convertScaleAbs(sobel_x_cv)
gy_display = cv2.convertScaleAbs(sobel_y_cv)
mag_display = np.clip(sobel_mag_cv, 0, 255).astype(np.uint8)

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Input Image (RGB)", fontsize=13, fontweight="bold")

axes[0, 1].imshow(sobel_x_cv, cmap="gray", vmin=-255, vmax=255)
axes[0, 1].set_title(
    "Horizontal Gradient G_x (Sobel X: Vertical Edges)", fontsize=13, fontweight="bold"
)

axes[1, 0].imshow(sobel_y_cv, cmap="gray", vmin=-255, vmax=255)
axes[1, 0].set_title(
    "Vertical Gradient G_y (Sobel Y: Horizontal Edges)", fontsize=13, fontweight="bold"
)

axes[1, 1].imshow(mag_display, cmap="gray")
axes[1, 1].set_title("Gradient Magnitude G = sqrt(G_x^2 + G_y^2)", fontsize=13, fontweight="bold")

for ax in axes.flatten():
    ax.axis("off")

plt.tight_layout()
plt.show()